In [ ]:
import os
import cv2
import numpy as np
import matplotlib
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score, jaccard_score

from skimage.feature import graycomatrix, graycoprops

In [ ]:
# 1. Загрузка и первичный просмотр данных

DATA_DIR = "KTH_TIPS"
RANDOM_STATE = 17


def load_texture_dataset(data_dir):
    images = []
    labels = []

    for cls in sorted(os.listdir(data_dir)):
        cls_path = os.path.join(data_dir, cls)
        if not os.path.isdir(cls_path):
            continue

        for fname in os.listdir(cls_path):
            if not fname.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif")):
                continue

            fpath = os.path.join(cls_path, fname)
            img = cv2.imread(fpath, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            img = cv2.resize(img, (128, 128))
            images.append(img)
            labels.append(cls)

    return np.array(images), np.array(labels)


X_raw, y_raw = load_texture_dataset(DATA_DIR)
print(f"Загружено изображений: {X_raw.shape[0]}")
print(f"Классы: {sorted(list(set(y_raw)))}")


def show_examples(images, labels, n=6):
    classes = sorted(list(set(labels)))[:n]
    plt.figure(figsize=(14, 3))

    for i, cls in enumerate(classes):
        idx = np.where(labels == cls)[0][0]
        img = images[idx]
        plt.subplot(1, n, i + 1)
        plt.imshow(img, cmap="gray")
        plt.title(cls)
        plt.axis("off")

    plt.suptitle("Примеры текстур")
    plt.tight_layout()
    plt.show()


show_examples(X_raw, y_raw, n=6)

Загружено изображений: 810

Классы: [np.str_('aluminium_foil'), np.str_('brown_bread'), np.str_('corduroy'), np.str_('cotton'), np.str_('cracker'), np.str_('linen'), np.str_('orange_peel'), np.str_('sandpaper'), np.str_('sponge'), np.str_('styrofoam')]

![Изображение](Figure_1.png)

In [ ]:
# 2. Признаки: Histogram

def hist_features(img, n_bins=24):
    hist = cv2.calcHist([img], [0], None, [n_bins], [0, 256]).flatten()
    hist = hist / (hist.sum() + 1e-7)

    mean = img.mean()
    std = img.std()
    p10 = np.percentile(img, 10)
    p90 = np.percentile(img, 90)

    return np.hstack([hist, mean, std, p10, p90])


def show_histogram_examples(images, labels, class_list=None, n_bins=24):
    if class_list is None:
        class_list = sorted(list(set(labels)))[:4]

    plt.figure(figsize=(14, 8))

    for i, cls_name in enumerate(class_list):
        idx = np.where(labels == cls_name)[0][0]
        img = images[idx]
        feats = hist_features(img)
        hist = feats[:n_bins]

        plt.subplot(len(class_list), 2, 2*i + 1)
        plt.imshow(img, cmap="gray")
        plt.title(f"{cls_name}")
        plt.axis("off")

        plt.subplot(len(class_list), 2, 2*i + 2)
        plt.bar(range(n_bins), hist)
        plt.title(f"Histogram ({cls_name})")
        plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


show_histogram_examples(X_raw, y_raw)


![Изображение](Figure_2.png)

In [ ]:
# 3. Признаки: Laws

def build_laws_kernels():
    L5 = np.array([1, 4, 6, 4, 1], dtype=np.float32)
    E5 = np.array([-1, -2, 0, 2, 1], dtype=np.float32)
    S5 = np.array([-1, 0, 2, 0, -1], dtype=np.float32)

    vecs = [L5, E5, S5]
    names = ["L", "E", "S"]

    kernels = {}
    for i in range(3):
        for j in range(3):
            kernels[f"{names[i]}5{names[j]}5"] = np.outer(vecs[i], vecs[j])

    return kernels


LAWS_KERNELS = build_laws_kernels()
LAWS_KERNEL_NAMES = list(LAWS_KERNELS.keys())


def laws_features(img):
    img_f = img.astype(np.float32)
    feats = []
    for k in LAWS_KERNELS.values():
        resp = cv2.filter2D(img_f, -1, k)
        feats.append(np.mean(np.abs(resp)))
    feats = np.array(feats)
    feats = feats / (feats.max() + 1e-7)
    return feats


def show_laws_examples(images, labels, class_list=None, top_k=5):
    if class_list is None:
        class_list = sorted(list(set(labels)))[:4]

    plt.figure(figsize=(8, 10))

    for i, cls_name in enumerate(class_list):
        idx = np.where(labels == cls_name)[0][0]
        img = images[idx]
        feats = laws_features(img)

        # сортируем по убыванию, берём top_k
        order = np.argsort(feats)[::-1][:top_k]

        # картинка
        plt.subplot(len(class_list), 2, 2 * i + 1)
        plt.imshow(img, cmap="gray")
        plt.title(f"{cls_name}\nOriginal")
        plt.axis("off")

        # текст
        plt.subplot(len(class_list), 2, 2 * i + 2)
        plt.axis("off")
        text_lines = ["Laws Texture Features", "", "Top Laws features:"]
        for j in order:
            text_lines.append(f"{LAWS_KERNEL_NAMES[j]}: {feats[j]:.4f}")
        plt.text(
            0.0,
            1.0,
            "\n".join(text_lines),
            va="top",
            ha="left",
            fontsize=9,
        )

    plt.tight_layout()
    plt.show()


show_laws_examples(X_raw, y_raw)

![Изображение](Figure_3.png)

In [ ]:
# 4. Признаки: GLCM

def glcm_features(img):
    img_q = (img / 16).astype(np.uint8)

    glcm = graycomatrix(
        img_q,
        distances=[1, 3],
        angles=[0, np.pi / 2],
        levels=16,
        symmetric=True,
        normed=True,
    )

    props = ["contrast", "dissimilarity", "homogeneity", "energy", "correlation"]
    values = [graycoprops(glcm, p).mean() for p in props]
    return np.array(values), props


def show_glcm_examples(images, labels, class_list=None):
    if class_list is None:
        class_list = sorted(list(set(labels)))[:4]

    plt.figure(figsize=(8, 10))

    for i, cls_name in enumerate(class_list):
        idx = np.where(labels == cls_name)[0][0]
        img = images[idx]
        feats, props = glcm_features(img)

        # картинка
        plt.subplot(len(class_list), 2, 2 * i + 1)
        plt.imshow(img, cmap="gray")
        plt.title(f"{cls_name}\nOriginal")
        plt.axis("off")

        # текст
        plt.subplot(len(class_list), 2, 2 * i + 2)
        plt.axis("off")
        lines = ["GLCM Texture Features", "", "GLCM Features:"]
        for name, val in zip(props, feats):
            lines.append(f"{name}: {val:.4f}")
        plt.text(
            0.0,
            1.0,
            "\n".join(lines),
            va="top",
            ha="left",
            fontsize=9,
        )

    plt.tight_layout()
    plt.show()


show_glcm_examples(X_raw, y_raw)

![Изображение](Figure_4.png)

In [ ]:
# 5. Формирование матриц признаков

def extract_all_features(images):
    F_hist, F_laws, F_glcm = [], [], []
    for img in images:
        F_hist.append(hist_features(img))
        F_laws.append(laws_features(img))
        F_glcm.append(glcm_features(img))
    return np.array(F_hist), np.array(F_laws), np.array(F_glcm)


X_hist, X_laws, X_glcm = extract_all_features(X_raw)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_raw)

# Классификация

def train_and_eval(X, y, name):
    print(f"   ПРИЗНАКИ: {name}")


    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
    )

    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train_sc = scaler.transform(X_train)
    X_test_sc = scaler.transform(X_test)

    # KNN
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train, y_train)
    pred_knn = knn.predict(X_test)
    print("\n--- KNN ---")
    print(classification_report(y_test, pred_knn, target_names=label_encoder.classes_))

    # SVM linear
    svm = SVC(kernel="linear", C=2)
    svm.fit(X_train_sc, y_train)
    pred_svm = svm.predict(X_test_sc)
    print("\n--- SVM (linear) ---")
    print(classification_report(y_test, pred_svm, target_names=label_encoder.classes_))

    # Decision Tree
    tree = DecisionTreeClassifier(max_depth=8)
    tree.fit(X_train, y_train)
    pred_tree = tree.predict(X_test)
    print("\n--- Decision Tree ---")
    print(classification_report(y_test, pred_tree, target_names=label_encoder.classes_))

    return {"KNN": knn, "SVM": (svm, scaler), "Tree": tree}


models_hist = train_and_eval(X_hist, y_encoded, "Histogram")
models_laws = train_and_eval(X_laws, y_encoded, "Laws")
models_glcm = train_and_eval(X_glcm, y_encoded, "GLCM")

In [ ]:
"""
ПРИЗНАКИ: Histogram

--- KNN ---
                precision    recall  f1-score   support

aluminium_foil       1.00      0.95      0.97        20
   brown_bread       0.59      0.81      0.68        21
      corduroy       0.55      0.55      0.55        20
        cotton       0.40      0.50      0.44        20
       cracker       0.63      0.60      0.62        20
         linen       0.60      0.60      0.60        20
   orange_peel       0.57      0.62      0.59        21
     sandpaper       0.26      0.25      0.26        20
        sponge       0.71      0.71      0.71        21
     styrofoam       0.62      0.25      0.36        20

      accuracy                           0.59       203
     macro avg       0.59      0.58      0.58       203
  weighted avg       0.59      0.59      0.58       203


--- SVM (linear) ---
                precision    recall  f1-score   support

aluminium_foil       1.00      1.00      1.00        20
   brown_bread       0.73      0.90      0.81        21
      corduroy       0.55      0.55      0.55        20
        cotton       0.39      0.45      0.42        20
       cracker       1.00      0.70      0.82        20
         linen       0.80      0.60      0.69        20
   orange_peel       0.81      0.81      0.81        21
     sandpaper       0.35      0.35      0.35        20
        sponge       0.64      0.86      0.73        21
     styrofoam       1.00      0.80      0.89        20

      accuracy                           0.70       203
     macro avg       0.73      0.70      0.71       203
  weighted avg       0.73      0.70      0.71       203


--- Decision Tree ---
                precision    recall  f1-score   support

aluminium_foil       1.00      1.00      1.00        20
   brown_bread       0.76      0.90      0.83        21
      corduroy       0.76      0.80      0.78        20
        cotton       0.53      0.40      0.46        20
       cracker       0.85      0.85      0.85        20
         linen       0.87      0.65      0.74        20
   orange_peel       0.89      0.81      0.85        21
     sandpaper       0.33      0.55      0.42        20
        sponge       0.83      0.71      0.77        21
     styrofoam       0.94      0.80      0.86        20

      accuracy                           0.75       203
     macro avg       0.78      0.75      0.76       203
  weighted avg       0.78      0.75      0.76       203


ПРИЗНАКИ: Laws

--- KNN ---
                precision    recall  f1-score   support

aluminium_foil       1.00      0.95      0.97        20
   brown_bread       0.67      0.67      0.67        21
      corduroy       1.00      0.80      0.89        20
        cotton       0.80      0.80      0.80        20
       cracker       0.50      0.80      0.62        20
         linen       0.85      0.55      0.67        20
   orange_peel       0.76      0.90      0.83        21
     sandpaper       0.40      0.20      0.27        20
        sponge       0.67      0.76      0.71        21
     styrofoam       0.65      0.75      0.70        20

      accuracy                           0.72       203
     macro avg       0.73      0.72      0.71       203
  weighted avg       0.73      0.72      0.71       203


--- SVM (linear) ---
                precision    recall  f1-score   support

aluminium_foil       0.95      1.00      0.98        20
   brown_bread       0.61      0.67      0.64        21
      corduroy       0.95      1.00      0.98        20
        cotton       0.94      0.75      0.83        20
       cracker       0.61      0.70      0.65        20
         linen       1.00      0.55      0.71        20
   orange_peel       0.55      1.00      0.71        21
     sandpaper       0.00      0.00      0.00        20
        sponge       0.79      0.52      0.63        21
     styrofoam       0.50      0.85      0.63        20

      accuracy                           0.70       203
     macro avg       0.69      0.70      0.68       203
  weighted avg       0.69      0.70      0.67       203


--- Decision Tree ---
                precision    recall  f1-score   support

aluminium_foil       1.00      0.90      0.95        20
   brown_bread       0.50      0.67      0.57        21
      corduroy       0.78      0.90      0.84        20
        cotton       0.88      0.75      0.81        20
       cracker       0.58      0.55      0.56        20
         linen       0.75      0.60      0.67        20
   orange_peel       0.62      0.86      0.72        21
     sandpaper       0.60      0.15      0.24        20
        sponge       0.56      0.48      0.51        21
     styrofoam       0.40      0.60      0.48        20

      accuracy                           0.65       203
     macro avg       0.67      0.64      0.64       203
  weighted avg       0.67      0.65      0.63       203



ПРИЗНАКИ: GLCM

--- KNN ---
                precision    recall  f1-score   support

aluminium_foil       0.94      0.75      0.83        20
   brown_bread       0.27      0.48      0.34        21
      corduroy       0.31      0.40      0.35        20
        cotton       0.25      0.20      0.22        20
       cracker       0.37      0.50      0.43        20
         linen       0.50      0.40      0.44        20
   orange_peel       0.76      0.62      0.68        21
     sandpaper       0.38      0.30      0.33        20
        sponge       0.47      0.38      0.42        21
     styrofoam       0.73      0.55      0.63        20

      accuracy                           0.46       203
     macro avg       0.50      0.46      0.47       203
  weighted avg       0.50      0.46      0.47       203


--- SVM (linear) ---
                precision    recall  f1-score   support

aluminium_foil       1.00      1.00      1.00        20
   brown_bread       0.25      0.29      0.27        21
      corduroy       0.23      0.25      0.24        20
        cotton       0.00      0.00      0.00        20
       cracker       0.59      0.65      0.62        20
         linen       0.48      0.75      0.59        20
   orange_peel       0.62      0.86      0.72        21
     sandpaper       0.27      0.20      0.23        20
        sponge       0.41      0.33      0.37        21
     styrofoam       0.64      0.70      0.67        20

      accuracy                           0.50       203
     macro avg       0.45      0.50      0.47       203
  weighted avg       0.45      0.50      0.47       203


--- Decision Tree ---
                precision    recall  f1-score   support

aluminium_foil       0.95      0.90      0.92        20
   brown_bread       0.32      0.67      0.43        21
      corduroy       0.29      0.20      0.24        20
        cotton       0.32      0.30      0.31        20
       cracker       0.58      0.55      0.56        20
         linen       0.60      0.45      0.51        20
   orange_peel       0.72      0.62      0.67        21
     sandpaper       0.42      0.40      0.41        20
        sponge       0.44      0.38      0.41        21
     styrofoam       0.78      0.70      0.74        20

      accuracy                           0.52       203
     macro avg       0.54      0.52      0.52       203
  weighted avg       0.54      0.52      0.52       203
"""

In [ ]:
# 7. Сегментация 3 моделями

selected_models = {
    "Histogram_KNN": ("hist", models_hist["KNN"]),
    "Laws_Tree": ("laws", models_laws["Tree"]),
    "Laws_SVM": ("laws", models_laws["SVM"]),
}

TARGET_IMAGE = "bread_gray.png"

def segment_image(model_info, image_path, patch_size=24, step=12):
    """
    Сегментация изображения:
      - окно patch_size x patch_size
      - шаг step
      - классификация каждого патча
    """
    feature_type, model = model_info

    img_color = cv2.imread(image_path)
    if img_color is None:
        print(f"нельзя загрузить {image_path}")
        return None, None

    img_gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)

    H, W = img_gray.shape
    seg_map = np.zeros((H, W), dtype=np.int32)

    scaler = None
    if isinstance(model, tuple):
        model, scaler = model

    for y in range(0, H - patch_size, step):
        for x in range(0, W - patch_size, step):
            patch = img_gray[y:y+patch_size, x:x+patch_size]

            if feature_type == "hist":
                feats = hist_features(patch)
            elif feature_type == "laws":
                feats = laws_features(patch)
            elif feature_type == "glcm":
                feats = glcm_features(patch)
            else:
                raise ValueError("Unknown feature type")

            feats = feats.reshape(1, -1)

            if scaler is not None:
                feats = scaler.transform(feats)

            cls = model.predict(feats)[0]
            seg_map[y:y+step, x:x+step] = cls

    return img_color, seg_map


def run_segmentation_for_all_models(target_image):
    imgs = []
    segs = []
    names = []

    for name, model_info in selected_models.items():
        print(f"\nМодель: {name}")

        img_color, seg = segment_image(
            model_info,
            target_image,
            patch_size=24,
            step=8
        )

        if seg is None:
            print("Ошибка сегментации!")
            continue

        # стандартизация классов
        uniq = np.unique(seg)
        print(f"Классы после фильтрации: {uniq}")

        seg_f = np.zeros_like(seg)
        for new_id, old_id in enumerate(uniq):
            seg_f[seg == old_id] = new_id

        print("Успешно")

        imgs.append(img_color)
        segs.append(seg_f)
        names.append(name)

    return imgs, segs, names


images_list, seg_maps_list, model_names = run_segmentation_for_all_models(TARGET_IMAGE)


plt.figure(figsize=(12, 4))

for i in range(len(seg_maps_list)):
    plt.subplot(1, 3, i+1)
    plt.imshow(seg_maps_list[i], cmap="tab20")
    plt.title(model_names[i])
    plt.axis("off")

plt.show()

print(f"\nУспешно моделей обработано: {len(seg_maps_list)}/3")


Модель: Histogram_KNN
Классы после фильтрации: [0 1 4 6]
Успешно

Модель: Laws_Tree
Классы после фильтрации: [0 1 2 3 4 5 6 7 8 9]
Успешно

Модель: Laws_SVM
Классы после фильтрации: [0 1 2 3 4 5 6 7 8 9]
Успешно

Успешно моделей обработано: 3/3

![Изображение](Figure_5.png)

In [ ]:
# Приведение ручной маски к 3 классам

def simplify_mask(mask, max_colors=3):
    unique, counts = np.unique(mask, return_counts=True)

    sorted_idx = np.argsort(counts)
    dominant = unique[sorted_idx][-max_colors:]

    print("Частые цвета:", dominant)

    result = np.zeros_like(mask)
    for new_id, orig in enumerate(dominant):
        result[mask == orig] = new_id

    print(f"Установлено: {list(dominant)} → {list(range(len(dominant)))}")
    return result

manual_mask = cv2.imread("bread_gray_mask.png", cv2.IMREAD_GRAYSCALE)
fixed_mask = simplify_mask(manual_mask, 3)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(manual_mask, cmap="gray")
plt.title(f"Исходная маска ({len(np.unique(manual_mask))} значений)")
plt.colorbar()

plt.subplot(1, 2, 2)
plt.imshow(fixed_mask, cmap="tab10")
plt.title("Исправленная маска (3 класса)")
plt.colorbar()

plt.show()

def compare_masks(manual_fixed, predicted, model_name, input_image):
    manual_classes = np.unique(manual_fixed)
    pred_classes = np.unique(predicted)

    print(f"Ручные классы: {manual_classes}")
    print(f"Классы предсказания: {pred_classes}")

    class_mapping = {
        0: 2,
        1: 1,
        2: 0,
    }

    pred_mapped = predicted.copy()
    for pred_cls, true_cls in class_mapping.items():
        pred_mapped[predicted == pred_cls] = true_cls

    acc = accuracy_score(manual_fixed.flatten(), pred_mapped.flatten())
    print(f"Точность: {acc*100:.1f}%")

    for cls in manual_classes:
        t = (manual_fixed == cls)
        p = (pred_mapped == cls)
        iou = jaccard_score(t.flatten(), p.flatten())
        print(f"Класс {cls}: IoU = {iou:.3f}")

    fig, ax = plt.subplots(1, 3, figsize=(20, 5))

    ax[0].imshow(input_image, cmap="gray")
    ax[0].set_title("Исходное изображение")
    ax[0].axis("off")

    ax[1].imshow(manual_fixed, cmap="tab10")
    ax[1].set_title("Ручная разметка")
    ax[1].axis("off")

    ax[2].imshow(pred_mapped, cmap="tab10")
    ax[2].set_title(f"Предсказание ({model_name})")
    ax[2].axis("off")

    plt.show()

    return acc


for name, seg_map in zip(model_names, seg_maps_list):

    if fixed_mask is not None and seg_map is not None:
        H, W = seg_map.shape
        manual_fixed_resized = cv2.resize(
            fixed_mask,
            (W, H),
            interpolation=cv2.INTER_NEAREST
        )
    else:
        print("Ошибка: нет маски или сегментации")
        continue

    acc = compare_masks(manual_fixed_resized, seg_map, name, images_list[0])
    print(f"Итог: {acc*100:.1f}% верных пикселей\n")

Частые цвета: [254   0 255]
Установлено: [np.uint8(254), np.uint8(0), np.uint8(255)] → [0, 1, 2]

![Изображение](Figure_6.png)


Ручные классы: [0 1 2]

Классы предсказания: [0 1 2 3]

Точность: 43.8%

Класс 0: IoU = 0.006

Класс 1: IoU = 0.728

Класс 2: IoU = 0.292

# Верных пикселей 43.8%

![Изображение](Figure_7.png)